# Symplectic Muon: NanoGPT-124M 3-Arm Bake-Off (Nesterov + Motifs Table)
**Project:** symplectic-muon
**Branch:** `feature/procedural-nanogpt-scale`
**Lead Researcher:** Jaime Jarauta
**One-Click Execution:** Press **Runtime -> Run all** to clone, setup, execute, sync to W&B, and auto-unassign GPU.

In [ ]:
# Step 1: Self-Healing Environment Setup & Credit Guard
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print('\n[Credit Guard] Unhandled error or interrupt! Syncing W&B and unassigning GPU...')
    try:
        subprocess.run('wandb sync --include-offline $(find /tmp/wandb ./wandb -type d -name "offline-run-*" 2>/dev/null) 2>/dev/null || true', shell=True)
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), lambda self, et, val, tb, tb_offset=None: (self.showtraceback((et, val, tb), tb_offset=tb_offset), auto_teardown()))

print('[Ready] Credit guard active. VM will self-terminate on error or completion.')


In [ ]:
# Step 2: Clone or Update Repository to Latest Feature Branch
import os, sys, subprocess

repo_name = 'symplectic-muon'
branch = 'feature/procedural-nanogpt-scale'
repo_url = f'https://github.com/jaimejarauta/symplectic-muon.git'

%cd /content
if not os.path.exists(f'/content/{repo_name}'):
    print(f'Cloning {repo_name} (branch: {branch})...')
    subprocess.run(['git', 'clone', '-b', branch, repo_url, f'/content/{repo_name}'], check=True)
else:
    print(f'Updating {repo_name} to latest {branch}...')
    subprocess.run(['git', '-C', f'/content/{repo_name}', 'fetch', 'origin', branch], check=True)
    subprocess.run(['git', '-C', f'/content/{repo_name}', 'checkout', branch], check=True)
    subprocess.run(['git', '-C', f'/content/{repo_name}', 'pull', 'origin', branch], check=True)

%cd /content/{repo_name}
!pip install -q -e . wandb pyyaml tiktoken

if f'/content/{repo_name}/src' not in sys.path:
    sys.path.insert(0, f'/content/{repo_name}/src')
os.environ['PYTHONPATH'] = f'/content/{repo_name}/src:' + os.environ.get('PYTHONPATH', '')

print(f'[Ready] {repo_name} installed in editable mode on branch {branch}.')


In [ ]:
# Step 3: Secure Weights & Biases Authentication (Non-blocking)
import os
wandb_key = os.environ.get('WANDB_API_KEY')
if not wandb_key:
    try:
        from google.colab import userdata
        for k in ['WANDB_API_KEY', 'wandb_key', 'WANDB_KEY', 'wandb', 'API_KEY']:
            try:
                val = userdata.get(k)
                if val and len(val.strip()) >= 20:
                    wandb_key = val.strip()
                    break
            except Exception:
                pass
    except Exception:
        pass

if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
    os.environ['WANDB_MODE'] = 'online'
    print('[W&B] Authenticated successfully in online mode.')
else:
    os.environ['WANDB_MODE'] = 'offline'
    print('[W&B] No API key detected. Running in offline mode (safe, never hangs).')


In [ ]:
# Step 4: Execute Long-Run Benchmark
!python scripts/run_colab.py


In [ ]:
# Step 5: Sync Telemetry & Unassign GPU (Zero Credit Leakage)
print('\n[Runner] Syncing all offline W&B runs to cloud...')
!wandb sync --include-offline $(find /tmp/wandb ./wandb -type d -name 'offline-run-*' 2>/dev/null) 2>/dev/null || true

print('\n[Credit Guard] Run complete! Unassigning Colab GPU immediately to save credits...')
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f'Teardown notice: {e}')
